In [1]:

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

SOURCE_ZIP = "/content/drive/MyDrive/AthletiQ/soccernet_ball_yolo.zip"
LOCAL_ZIP = "/content/soccernet_ball_yolo.zip"

assert os.path.exists(SOURCE_ZIP), f"Dataset ZIP not found: {SOURCE_ZIP}"

if not os.path.exists(LOCAL_ZIP) or (
    os.path.getsize(LOCAL_ZIP) != os.path.getsize(SOURCE_ZIP)
):
    print("Copying dataset ZIP from Drive to Colab...")
    shutil.copy2(SOURCE_ZIP, LOCAL_ZIP)
else:
    print("A ZIP of matching size already exists in Colab; skipping copy.")

print(f"ZIP size: {os.path.getsize(LOCAL_ZIP) / (1024**3):.2f} GB")
print("Copy complete.")


Mounted at /content/drive
Copying dataset ZIP from Drive to Colab...
ZIP size: 17.05 GB
Copy complete.


In [2]:

import os
import zipfile

LOCAL_ZIP = "/content/soccernet_ball_yolo.zip"
EXTRACT_DIR = "/content/soccernet_ball_yolo"
DATA_YAML = os.path.join(
    EXTRACT_DIR, "soccernet_ball_yolo", "data.yaml"
)

if os.path.exists(DATA_YAML):
    print("Dataset already extracted. Skipping extraction.")
else:
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        bad_file = zf.testzip()
        if bad_file is not None:
            raise RuntimeError(f"Corrupt ZIP member: {bad_file}")

        print(f"Extracting {len(zf.infolist())} archive entries...")
        zf.extractall(EXTRACT_DIR)

assert os.path.exists(DATA_YAML), (
    f"Expected data.yaml not found: {DATA_YAML}"
)

print("Extraction complete.")
print("Dataset YAML:", DATA_YAML)
print("Dataset directory:", os.path.dirname(DATA_YAML))


Extracting 159011 archive entries...
Extraction complete.
Dataset YAML: /content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml
Dataset directory: /content/soccernet_ball_yolo/soccernet_ball_yolo


In [3]:

import os
import yaml

DATA_YAML = "/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml"
CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("Dataset configuration:")
print(config)

print("\nChampion checkpoint exists:", os.path.exists(CHAMPION))
if os.path.exists(CHAMPION):
    print(f"Champion size: {os.path.getsize(CHAMPION) / (1024**2):.2f} MB")


Dataset configuration:
{'path': '/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/processed/soccernet_ball_yolo', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'ball'}}

Champion checkpoint exists: True
Champion size: 5.27 MB


In [4]:

import os
import yaml

DATA_YAML = "/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml"
DATASET_DIR = os.path.dirname(DATA_YAML)

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

# Fix the old Windows path for Colab
config["path"] = DATASET_DIR

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Updated dataset path:", config["path"])

for split in ["train", "val", "test"]:
    image_dir = os.path.join(DATASET_DIR, config[split])
    label_dir = os.path.join(DATASET_DIR, "labels", split)

    print(f"\n{split.upper()}")
    print("Images folder exists:", os.path.isdir(image_dir))
    print("Labels folder exists:", os.path.isdir(label_dir))

    if os.path.isdir(image_dir):
        print("Image count:", sum(
            f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
            for f in os.listdir(image_dir)
        ))

print("\nClass names:", config["names"])


Updated dataset path: /content/soccernet_ball_yolo/soccernet_ball_yolo

TRAIN
Images folder exists: True
Labels folder exists: True
Image count: 55500

VAL
Images folder exists: True
Labels folder exists: True
Image count: 11250

TEST
Images folder exists: True
Labels folder exists: True
Image count: 12750

Class names: {0: 'ball'}


In [6]:
!pip install -q ultralytics

import os
import torch
import yaml
from ultralytics import YOLO

DATA_YAML = "/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml"
CHAMPION = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep/weights/best.pt"
)

assert os.path.isfile(DATA_YAML), "Dataset YAML missing"
assert os.path.isfile(CHAMPION), "Champion checkpoint missing"

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

print("Ultralytics model loading...")
model = YOLO(CHAMPION)

print("\nModel loaded successfully.")
print("Checkpoint:", CHAMPION)
print("Dataset:", cfg["path"])
print("Class names:", model.names)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 10.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
Ultralytics model loading...

Model loaded successfully.
Checkpoint: /content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/best.pt
Dataset: /content/soccernet_ball_yolo/soccernet_ball_yolo
Class names: {0: 'ball'}


In [7]:

import os
import csv
import yaml
import numpy as np
from tqdm.auto import tqdm

DATA_YAML = "/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml"
OUTPUT_DIR = "/content/athletiq_hard_mining"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

ROOT = cfg["path"]
CONF = 0.10
IMGSZ = 1280
BATCH_SIZE = 16
TOP_K = 1500

def read_gt(label_path):
    boxes = []
    if os.path.isfile(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls, xc, yc, w, h = map(float, parts[:5])
                    if int(cls) == 0:
                        boxes.append([
                            xc - w/2, yc - h/2,
                            xc + w/2, yc + h/2
                        ])
    return boxes

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    aa = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    ab = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = aa + ab - inter
    return inter / union if union > 0 else 0.0

def score_image(gt, pred):
    pairs = []
    for gi, g in enumerate(gt):
        for pi, p in enumerate(pred):
            v = iou(g, p)
            if v >= 0.50:
                pairs.append((v, gi, pi))
    pairs.sort(reverse=True)

    used_gt, used_pred = set(), set()
    for _, gi, pi in pairs:
        if gi not in used_gt and pi not in used_pred:
            used_gt.add(gi)
            used_pred.add(pi)

    missed = len(gt) - len(used_gt)
    false_pos = len(pred) - len(used_pred)
    # Prioritize misses while also retaining false-positive examples.
    hard_score = 3 * missed + 2 * false_pos
    return missed, false_pos, hard_score

for split in ["train", "val", "test"]:
    image_dir = os.path.join(ROOT, "images", split)
    label_dir = os.path.join(ROOT, "labels", split)
    csv_path = os.path.join(OUTPUT_DIR, f"{split}_hard_ranking.csv")
    top_path = os.path.join(OUTPUT_DIR, f"{split}_top1500.csv")

    images = sorted(
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    )

    # Reuse a completed ranking only when its image inventory matches.
    cached = []
    if os.path.isfile(csv_path):
        with open(csv_path, newline="") as f:
            cached = list(csv.DictReader(f))

    if len(cached) == len(images) and set(r["image"] for r in cached) == set(images):
        print(f"\n[{split.upper()}] Reusing existing full ranking.")
        ranked = sorted(cached, key=lambda r: float(r["hard_score"]), reverse=True)
    else:
        rows = []
        for start in tqdm(range(0, len(images), BATCH_SIZE), desc=f"Mining {split}"):
            batch_names = images[start:start+BATCH_SIZE]
            batch_paths = [os.path.join(image_dir, name) for name in batch_names]

            results = model.predict(
                source=batch_paths,
                imgsz=IMGSZ,
                conf=CONF,
                max_det=50,
                batch=BATCH_SIZE,
                device=0,
                verbose=False
            )

            for name, result in zip(batch_names, results):
                label_path = os.path.join(
                    label_dir, os.path.splitext(name)[0] + ".txt"
                )
                gt = read_gt(label_path)
                pred = (
                    result.boxes.xyxy.cpu().numpy().tolist()
                    if result.boxes is not None and len(result.boxes)
                    else []
                )

                missed, false_pos, hard_score = score_image(gt, pred)
                rows.append({
                    "split": split,
                    "image": name,
                    "label": label_path,
                    "gt_balls": len(gt),
                    "predictions": len(pred),
                    "missed_balls": missed,
                    "false_positives": false_pos,
                    "hard_score": hard_score
                })

        ranked = sorted(rows, key=lambda r: int(r["hard_score"]), reverse=True)
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=ranked[0].keys())
            writer.writeheader()
            writer.writerows(ranked)

    top = ranked[:min(TOP_K, len(ranked))]
    with open(top_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=top[0].keys())
        writer.writeheader()
        writer.writerows(top)

    print(f"\n[{split.upper()}]")
    print("Images scored:", len(ranked))
    print("Hard candidates saved:", len(top))
    print("Full ranking:", csv_path)
    print("Top candidates:", top_path)
    print("Top 5:")
    for row in top[:5]:
        print(row["image"],
              "missed =", row["missed_balls"],
              "false positives =", row["false_positives"],
              "score =", row["hard_score"])

print("\nMining all splits complete.")


Mining train:   0%|          | 0/3469 [00:00<?, ?it/s]


[TRAIN]
Images scored: 55500
Hard candidates saved: 1500
Full ranking: /content/athletiq_hard_mining/train_hard_ranking.csv
Top candidates: /content/athletiq_hard_mining/train_top1500.csv
Top 5:
SNMOT-136_000646.jpg missed = 2 false positives = 7 score = 20
SNMOT-136_000664.jpg missed = 2 false positives = 7 score = 20
SNMOT-136_000657.jpg missed = 2 false positives = 6 score = 18
SNMOT-136_000658.jpg missed = 2 false positives = 6 score = 18
SNMOT-169_000080.jpg missed = 2 false positives = 6 score = 18


Mining val:   0%|          | 0/704 [00:00<?, ?it/s]


[VAL]
Images scored: 11250
Hard candidates saved: 1500
Full ranking: /content/athletiq_hard_mining/val_hard_ranking.csv
Top candidates: /content/athletiq_hard_mining/val_top1500.csv
Top 5:
SNMOT-164_000077.jpg missed = 2 false positives = 5 score = 16
SNMOT-107_000377.jpg missed = 1 false positives = 6 score = 15
SNMOT-164_000034.jpg missed = 2 false positives = 4 score = 14
SNMOT-164_000063.jpg missed = 2 false positives = 4 score = 14
SNMOT-164_000082.jpg missed = 2 false positives = 4 score = 14


Mining test:   0%|          | 0/797 [00:00<?, ?it/s]


[TEST]
Images scored: 12750
Hard candidates saved: 1500
Full ranking: /content/athletiq_hard_mining/test_hard_ranking.csv
Top candidates: /content/athletiq_hard_mining/test_top1500.csv
Top 5:
SNMOT-105_000621.jpg missed = 2 false positives = 6 score = 18
SNMOT-105_000622.jpg missed = 2 false positives = 6 score = 18
SNMOT-105_000642.jpg missed = 2 false positives = 6 score = 18
SNMOT-105_000620.jpg missed = 2 false positives = 5 score = 16
SNMOT-105_000641.jpg missed = 2 false positives = 5 score = 16

Mining all splits complete.


In [8]:

import os
import csv
import shutil
import zipfile
from pathlib import Path

# ---------- PATHS ----------
DRIVE_ROOT = "/content/drive/MyDrive/AthletiQ"
DATASET_ROOT = "/content/soccernet_ball_yolo/soccernet_ball_yolo"
MINING_DIR = "/content/athletiq_hard_mining"

ORIGINAL_ZIP = os.path.join(DRIVE_ROOT, "soccernet_ball_yolo.zip")
LOCAL_ARCHIVE = "/content/hard_mined_soccernet_ball_yolo.zip"
DRIVE_ARCHIVE = os.path.join(
    DRIVE_ROOT, "hard_mined_soccernet_ball_yolo.zip"
)
STAGING_DIR = "/content/hard_mined_soccernet_ball_yolo"

SPLITS = ["train", "val", "test"]
EXPECTED_PER_SPLIT = 1500

# ---------- SAFETY CHECKS ----------
assert os.path.isdir(DRIVE_ROOT), "Google Drive is not mounted."
assert os.path.isdir(DATASET_ROOT), "Extracted dataset not found."

for split in SPLITS:
    csv_path = os.path.join(MINING_DIR, f"{split}_top1500.csv")
    assert os.path.isfile(csv_path), f"Missing mined selection: {csv_path}"

# Read selections and verify each has exactly 1,500 unique images.
selections = {}

for split in SPLITS:
    csv_path = os.path.join(MINING_DIR, f"{split}_top1500.csv")
    with open(csv_path, newline="") as f:
        rows = list(csv.DictReader(f))

    assert len(rows) == EXPECTED_PER_SPLIT, (
        f"{split}: expected {EXPECTED_PER_SPLIT} rows, found {len(rows)}"
    )

    names = [r["image"] for r in rows]
    assert len(set(names)) == EXPECTED_PER_SPLIT, (
        f"{split} contains duplicate image names."
    )
    selections[split] = rows

print("All three selections verified: 1,500 images each.")

# ---------- PREPARE STAGING DIRECTORY ----------
if os.path.exists(STAGING_DIR):
    shutil.rmtree(STAGING_DIR)

os.makedirs(STAGING_DIR)

manifest_path = os.path.join(STAGING_DIR, "manifest.csv")
manifest_fields = [
    "split", "image", "label", "gt_balls",
    "predictions", "missed_balls", "false_positives", "hard_score"
]

total_images = 0

with open(manifest_path, "w", newline="") as manifest_file:
    writer = csv.DictWriter(manifest_file, fieldnames=manifest_fields)
    writer.writeheader()

    for split in SPLITS:
        source_images = os.path.join(DATASET_ROOT, "images", split)
        source_labels = os.path.join(DATASET_ROOT, "labels", split)

        dest_images = os.path.join(STAGING_DIR, split, "images")
        dest_labels = os.path.join(STAGING_DIR, split, "labels")

        os.makedirs(dest_images, exist_ok=True)
        os.makedirs(dest_labels, exist_ok=True)

        for row in selections[split]:
            image_name = row["image"]
            stem = os.path.splitext(image_name)[0]

            src_image = os.path.join(source_images, image_name)
            src_label = os.path.join(source_labels, stem + ".txt")

            dst_image = os.path.join(dest_images, image_name)
            dst_label = os.path.join(dest_labels, stem + ".txt")

            assert os.path.isfile(src_image), (
                f"Missing source image: {src_image}"
            )

            shutil.copy2(src_image, dst_image)

            # Background images may legitimately have no label file.
            # Create an empty YOLO label file for those images.
            if os.path.isfile(src_label):
                shutil.copy2(src_label, dst_label)
            else:
                open(dst_label, "w").close()

            writer.writerow({
                "split": split,
                "image": image_name,
                "label": f"{split}/labels/{stem}.txt",
                "gt_balls": row.get("gt_balls", ""),
                "predictions": row.get("predictions", ""),
                "missed_balls": row.get("missed_balls", ""),
                "false_positives": row.get("false_positives", ""),
                "hard_score": row.get("hard_score", "")
            })

            total_images += 1

        # Preserve the complete mining ranking for this split.
        shutil.copy2(
            os.path.join(MINING_DIR, f"{split}_hard_ranking.csv"),
            os.path.join(STAGING_DIR, f"{split}_hard_ranking.csv")
        )

print(f"Prepared {total_images} images and matching label files.")

# ---------- CREATE LOCAL ZIP ----------
if os.path.exists(LOCAL_ARCHIVE):
    os.remove(LOCAL_ARCHIVE)

with zipfile.ZipFile(
    LOCAL_ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for root, _, files in os.walk(STAGING_DIR):
        for filename in files:
            path = os.path.join(root, filename)
            arcname = os.path.relpath(path, STAGING_DIR)
            zf.write(path, arcname)

# Verify local ZIP integrity and expected contents.
with zipfile.ZipFile(LOCAL_ARCHIVE, "r") as zf:
    assert zf.testzip() is None, "Local ZIP integrity check failed."
    names = zf.namelist()

    image_entries = [
        n for n in names
        if "/images/" in n and n.lower().endswith(
            (".jpg", ".jpeg", ".png", ".webp")
        )
    ]
    label_entries = [n for n in names if "/labels/" in n and n.endswith(".txt")]

    assert len(image_entries) == 4500, (
        f"Expected 4,500 images in archive, found {len(image_entries)}"
    )
    assert len(label_entries) == 4500, (
        f"Expected 4,500 label files, found {len(label_entries)}"
    )

print("Local archive verified.")
print(f"Archive size: {os.path.getsize(LOCAL_ARCHIVE) / (1024**3):.2f} GB")

# ---------- COPY TO DRIVE AND VERIFY ----------
shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)

assert os.path.isfile(DRIVE_ARCHIVE), "Archive copy to Drive failed."
assert os.path.getsize(DRIVE_ARCHIVE) == os.path.getsize(LOCAL_ARCHIVE), (
    "Drive archive size differs from local archive."
)

with zipfile.ZipFile(DRIVE_ARCHIVE, "r") as zf:
    assert zf.testzip() is None, "Drive archive integrity check failed."
    assert len([
        n for n in zf.namelist()
        if "/images/" in n and n.lower().endswith(
            (".jpg", ".jpeg", ".png", ".webp")
        )
    ]) == 4500, "Drive archive image count is incorrect."

print("\nSUCCESS: Hard-mined dataset archive verified in Drive.")
print("Saved to:", DRIVE_ARCHIVE)

# ---------- DELETE ORIGINAL ZIP ONLY AFTER VERIFICATION ----------
if os.path.isfile(ORIGINAL_ZIP):
    os.remove(ORIGINAL_ZIP)
    print("Original dataset ZIP deleted from Drive:", ORIGINAL_ZIP)
else:
    print("Original dataset ZIP was already absent.")

print("\nFinal archive remains in Drive.")
print("Train: 1,500 | Validation: 1,500 | Test: 1,500")


All three selections verified: 1,500 images each.
Prepared 4500 images and matching label files.
Local archive verified.
Archive size: 0.96 GB

SUCCESS: Hard-mined dataset archive verified in Drive.
Saved to: /content/drive/MyDrive/AthletiQ/hard_mined_soccernet_ball_yolo.zip
Original dataset ZIP deleted from Drive: /content/drive/MyDrive/AthletiQ/soccernet_ball_yolo.zip

Final archive remains in Drive.
Train: 1,500 | Validation: 1,500 | Test: 1,500
